<a href="https://colab.research.google.com/github/stevenolanecon/7002LBSAI/blob/main/notebooks/week9_collider_simulation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 9 – the movie-star collider

Why do so many good-looking film stars seem to be poor actors? This notebook runs the simulation from the workshop step by step. Every number on the slides comes from here.

No dataset to load: we make the data ourselves, so we know the truth.

**Before you start:**

- Go to **File → Save a copy in Drive**. Until you do this, your work only exists in this browser tab – you can't save changes back to GitHub, and closing the tab or losing the session will lose your work.
- Turn off Colab's autocomplete: **Tools → Settings → Editor → uncheck "Show context-powered code completions."** It's a setting on your own Google account, not something built into this notebook, so you'll need to do it once per account.

## Setup

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt

## Step 1 – 1,000 actors, talent and looks unrelated

Each actor gets a talent score and a looks score, both averaging 50 with a standard deviation of 10. They're drawn separately, so they're unrelated by construction. The seed fixes the random draws so you get the same numbers as the slides.

In [ ]:
rng = np.random.default_rng(2)
talent = rng.normal(50, 10, 1000)
looks = rng.normal(50, 10, 1000)
luck = rng.normal(0, 5, 1000)
actors = pd.DataFrame({"talent": talent, "looks": looks})
print("Correlation between talent and looks:", round(actors["talent"].corr(actors["looks"]), 3))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.scatter(actors["looks"], actors["talent"], s=8, color="grey")
all_fit = smf.ols("talent ~ looks", data=actors).fit(cov_type="HC1")
xs = np.linspace(actors["looks"].min(), actors["looks"].max(), 50)
ax.plot(xs, all_fit.params["Intercept"] + all_fit.params["looks"] * xs, color="black")
ax.set_xlabel("Looks (score)")
ax.set_ylabel("Acting talent (score)")
plt.show()

## Step 2 – who becomes a star?

An actor becomes a star if talent + looks + a bit of luck is above 110.

In [ ]:
actors["star"] = ((talent + looks + luck) > 110).astype(int)
print(actors["star"].sum(), "stars out of", len(actors))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.scatter(actors["looks"], actors["talent"], s=8,
           c=np.where(actors["star"] == 1, "red", "lightgrey"))
ax.plot(xs, 110 - xs, color="black", linestyle="--", label="talent + looks = 110")
ax.set_ylim(actors["talent"].min(), actors["talent"].max())
ax.set_xlabel("Looks (score)")
ax.set_ylabel("Acting talent (score)")
ax.legend()
plt.show()

## Step 3 – look only at the stars

In [ ]:
stars = actors[actors["star"] == 1]
star_fit = smf.ols("talent ~ looks", data=stars).fit(cov_type="HC1")
fig, ax = plt.subplots(figsize=(8, 4))
ax.scatter(stars["looks"], stars["talent"], s=8, color="red")
xs_s = np.linspace(stars["looks"].min(), stars["looks"].max(), 50)
ax.plot(xs_s, star_fit.params["Intercept"] + star_fit.params["looks"] * xs_s, color="red")
ax.plot(xs, all_fit.params["Intercept"] + all_fit.params["looks"] * xs, color="black", alpha=0.5)
ax.set_xlabel("Looks (score)")
ax.set_ylabel("Acting talent (score)")
plt.show()

## Two ways to fall into it

Picking the sample on the collider, or adding it as a control, both create a relationship that isn't there.

In [ ]:
ctrl_fit = smf.ols("talent ~ looks + star", data=actors).fit(cov_type="HC1")
pd.DataFrame({
    "Coefficient on looks": [all_fit.params["looks"], star_fit.params["looks"], ctrl_fit.params["looks"]],
    "p-value": [all_fit.pvalues["looks"], star_fit.pvalues["looks"], ctrl_fit.pvalues["looks"]],
}, index=["All actors", "Stars only", "All actors, controlling for being a star"]).round(3)

## Try it yourself

The function below runs the whole simulation in one go. Change the inputs and see what happens to the stars-only coefficient:

- a **stricter** cut-off (say 120): fewer stars, and is the bias bigger or smaller?
- **more luck** (say `luck_sd=20`): stardom depends less on talent and looks – what happens to the bias?
- **no luck** (`luck_sd=0`)
- a **bigger** sample (`n=10000`): does the bias go away? (It shouldn't – more data doesn't fix a collider.)

In [ ]:
def simulate(cutoff=110, luck_sd=5, n=1000, seed=2):
    rng = np.random.default_rng(seed)
    t = rng.normal(50, 10, n)
    l = rng.normal(50, 10, n)
    df = pd.DataFrame({"talent": t, "looks": l,
                       "star": ((t + l + rng.normal(0, luck_sd, n)) > cutoff).astype(int)})
    all_b = smf.ols("talent ~ looks", data=df).fit().params["looks"]
    star_b = smf.ols("talent ~ looks", data=df[df["star"] == 1]).fit().params["looks"]
    return {"stars": int(df["star"].sum()), "all actors": round(float(all_b), 2), "stars only": round(float(star_b), 2)}

simulate()

In [ ]:
simulate(cutoff=120)